# Hyperparameter Tunning with MLFlow and Optuna

We will do hyperparameter tuning for the LightGBM model because it performed better in 2 out of the three evaluated metrics: `MAE`, `RMSE`, and `R2`.

In [17]:
# Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import mlflow
import mlflow.lightgbm

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

In [18]:
# Load datasets
train_df = pd.read_csv("/Users/mateopozoruiz/Projects/Housing-Pricing-End-to-End-ML/data/processed/feature_engineered_train_data.csv")
test_df = pd.read_csv("/Users/mateopozoruiz/Projects/Housing-Pricing-End-to-End-ML/data/processed/feature_engineered_test_data.csv")

In [19]:
# Define features and target
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_test, y_test = test_df.drop(columns=[target]), test_df[target]

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (576815, 39)
Testing data shape: (148448, 39)


In [20]:
# Define Optuna objective function
def objective(trial: optuna.trial.Trial) -> float:
    """
    Objective function for Optuna hyperparameter tuning.

    Args:
        trial (optuna.trial.Trial): An Optuna trial object.

    Returns:
        float: The RMSE of the model on the test set.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 3000),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }

    with mlflow.start_run():
        model = LGBMRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Log parameters and metrics to MLflow
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [21]:
# Run Optuna study with MLFlow tracking
mlflow.set_experiment("LightGBM_Hyperparameter_Tuning")
mlflow.set_tracking_uri("/Users/mateopozoruiz/Projects/Housing-Pricing-End-to-End-ML/mlruns")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best trial:", study.best_trial.params)

[I 2026-01-19 22:12:41,035] A new study created in memory with name: no-name-ee7c374c-c78c-40e2-b08d-35f48d8abe71


[LightGBM] [Warning] lambda_l1 is set=2.437228469285163, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.437228469285163
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.42304498325866274, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.42304498325866274
[LightGBM] [Warning] bagging_fraction is set=0.790416611784803, subsample=1.0 will be ignored. Current value: bagging_fraction=0.790416611784803
[LightGBM] [Warning] lambda_l2 is set=7.35847098050742, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.35847098050742
[LightGBM] [Warning] min_data_in_leaf is set=84, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=84
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=2.437228469285163, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.4372284692851

[I 2026-01-19 22:12:46,489] Trial 0 finished with value: 76788.97386785086 and parameters: {'n_estimators': 743, 'learning_rate': 0.02501479861524213, 'num_leaves': 427, 'max_depth': 4, 'min_data_in_leaf': 84, 'feature_fraction': 0.42304498325866274, 'bagging_fraction': 0.790416611784803, 'bagging_freq': 5, 'lambda_l1': 2.437228469285163, 'lambda_l2': 7.35847098050742, 'min_split_gain': 0.2245769809170589}. Best is trial 0 with value: 76788.97386785086.


[LightGBM] [Warning] lambda_l1 is set=3.111106865357388, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.111106865357388
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] feature_fraction is set=0.942719206706082, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.942719206706082
[LightGBM] [Warning] bagging_fraction is set=0.7132063224013392, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7132063224013392
[LightGBM] [Warning] lambda_l2 is set=8.514079076126846, reg_lambda=0.0 will be ignored. Current value: lambda_l2=8.514079076126846
[LightGBM] [Warning] min_data_in_leaf is set=42, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=42
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=3.111106865357388, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.1111068653573

[I 2026-01-19 22:12:49,988] Trial 1 finished with value: 73840.85373690973 and parameters: {'n_estimators': 606, 'learning_rate': 0.11375179138581018, 'num_leaves': 2623, 'max_depth': 3, 'min_data_in_leaf': 42, 'feature_fraction': 0.942719206706082, 'bagging_fraction': 0.7132063224013392, 'bagging_freq': 6, 'lambda_l1': 3.111106865357388, 'lambda_l2': 8.514079076126846, 'min_split_gain': 0.6535131703949164}. Best is trial 1 with value: 73840.85373690973.


...

[LightGBM] [Warning] lambda_l1 is set=5.147053942608003, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5.147053942608003
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6023513642416257, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6023513642416257
[LightGBM] [Warning] bagging_fraction is set=0.6247735292355396, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6247735292355396
[LightGBM] [Warning] lambda_l2 is set=4.410167870346035, reg_lambda=0.0 will be ignored. Current value: lambda_l2=4.410167870346035
[LightGBM] [Warning] min_data_in_leaf is set=70, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=70
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=5.147053942608003, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5.14705394260

[I 2026-01-19 22:37:04,135] Trial 48 finished with value: 66940.01150423633 and parameters: {'n_estimators': 769, 'learning_rate': 0.05439296345961537, 'num_leaves': 2384, 'max_depth': 6, 'min_data_in_leaf': 70, 'feature_fraction': 0.6023513642416257, 'bagging_fraction': 0.6247735292355396, 'bagging_freq': 3, 'lambda_l1': 5.147053942608003, 'lambda_l2': 4.410167870346035, 'min_split_gain': 0.5239207500570686}. Best is trial 39 with value: 65180.47585219112.


[LightGBM] [Warning] lambda_l1 is set=4.832715988717269, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.832715988717269
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.5167967166014578, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5167967166014578
[LightGBM] [Warning] bagging_fraction is set=0.7592264485408312, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7592264485408312
[LightGBM] [Warning] lambda_l2 is set=2.4598990028392307, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.4598990028392307
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=4.832715988717269, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.832715988

[I 2026-01-19 22:37:17,115] Trial 49 finished with value: 67170.29623398591 and parameters: {'n_estimators': 650, 'learning_rate': 0.12352959921380298, 'num_leaves': 1689, 'max_depth': 7, 'min_data_in_leaf': 50, 'feature_fraction': 0.5167967166014578, 'bagging_fraction': 0.7592264485408312, 'bagging_freq': 4, 'lambda_l1': 4.832715988717269, 'lambda_l2': 2.4598990028392307, 'min_split_gain': 0.3982374546462098}. Best is trial 39 with value: 65180.47585219112.


Best trial: {'n_estimators': 638, 'learning_rate': 0.04390859550273521, 'num_leaves': 485, 'max_depth': 9, 'min_data_in_leaf': 58, 'feature_fraction': 0.6209477265151152, 'bagging_fraction': 0.7387286657442765, 'bagging_freq': 3, 'lambda_l1': 3.3256740893639734, 'lambda_l2': 3.27304543391332, 'min_split_gain': 0.30438359312123664}


In [22]:
# Train model with best hyperparameters
best_params = study.best_trial.params
best_model = LGBMRegressor(**best_params)
best_model.fit(X_train, y_train)

# Evaluate the best model
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Best Model Performance:\nRMSE: {rmse}\nMAE: {mae}\nR2: {r2}")

[LightGBM] [Warning] lambda_l1 is set=3.3256740893639734, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.3256740893639734
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6209477265151152, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6209477265151152
[LightGBM] [Warning] bagging_fraction is set=0.7387286657442765, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7387286657442765
[LightGBM] [Warning] lambda_l2 is set=3.27304543391332, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.27304543391332
[LightGBM] [Warning] min_data_in_leaf is set=58, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=58
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=3.3256740893639734, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.3256740893

In [23]:
# Log the best model to MLflow
with mlflow.start_run(run_name="Best-LightGBM-Model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.lightgbm.log_model(best_model, name="model")

2026/01/19 22:45:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
